In [8]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

plt.rcParams["font.family"] = "serif"

df_kmeans = pd.read_csv("rekapan_evaluasi_KMEANS.csv")
df_fcm = pd.read_csv("rekapan_evaluasi_FCM.csv")

df_combined = pd.concat([df_kmeans, df_fcm], ignore_index=True)

df_grouped = (
    df_combined
    .groupby(["Algoritma", "Metrik Jarak"])[["Akurasi (%)", "Waktu Eksekusi (s)", "Silhouette Score"]]
    .mean()
    .reset_index()
)

label_map = {
    ("K-Means", "Euclidean"): "K-Means\nEuclidean",
    ("K-Means", "Manhattan"): "K-Means\nManhattan",
    ("K-Means", "Minkowski"): "K-Means\nMinkowski",
    ("FCM", "Euclidean (default FCM)"): "FCM",
}

df_grouped["Label"] = df_grouped.apply(
    lambda row: label_map.get((row["Algoritma"], row["Metrik Jarak"]), row["Metrik Jarak"]),
    axis=1,
)

desired_order = [
    "K-Means\nEuclidean",
    "K-Means\nManhattan",
    "K-Means\nMinkowski",
    "FCM",
]

df_grouped["LabelOrder"] = df_grouped["Label"].apply(
    lambda x: desired_order.index(x) if x in desired_order else 999
)
df_grouped = df_grouped.sort_values("LabelOrder").reset_index(drop=True)

bar_colors = ["#3168A6", "#9E2A2B", "#2E7D32", "#616161"]

fig1, ax1 = plt.subplots(figsize=(10, 6))

bars = ax1.bar(
    df_grouped["Label"],
    df_grouped["Akurasi (%)"],
    color=bar_colors,
    width=0.5,
    edgecolor="black",
    linewidth=0.8,
)

for bar, val in zip(bars, df_grouped["Akurasi (%)"]):
    ax1.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.4,
        f"{val:.2f}%",
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold",
        color="#222222",
    )

ax1.set_title(
    "Komparasi Rata-rata Akurasi per Algoritma & Metrik Jarak",
    fontsize=14,
    fontweight="bold",
    pad=14,
)
ax1.set_xlabel("Algoritma & Metrik Jarak", fontsize=12, labelpad=8)
ax1.set_ylabel("Rata-rata Akurasi (%)", fontsize=12, labelpad=8)

y_min = max(0, df_grouped["Akurasi (%)"].min() - 5)
y_max = df_grouped["Akurasi (%)"].max() + 4
ax1.set_ylim(y_min, y_max)

ax1.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.1f%%"))
ax1.tick_params(axis="x", labelsize=11)
ax1.tick_params(axis="y", labelsize=10)
ax1.grid(axis="y", linestyle="--", alpha=0.4, color="gray")
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)

fig1.tight_layout()
fig1.savefig("Grafik_Akurasi.png", dpi=300, bbox_inches="tight")
plt.close(fig1)

scatter_colors = bar_colors
scatter_labels = [
    "K-Means Euclidean",
    "K-Means Manhattan",
    "K-Means Minkowski",
    "FCM",
]
label_to_scatter = {
    "K-Means\nEuclidean": "K-Means Euclidean",
    "K-Means\nManhattan": "K-Means Manhattan",
    "K-Means\nMinkowski": "K-Means Minkowski",
    "FCM": "FCM",
}

fig2, ax2 = plt.subplots(figsize=(10, 7))

for idx, row in df_grouped.iterrows():
    sc_label = label_to_scatter.get(row["Label"], row["Label"])
    ax2.scatter(
        row["Waktu Eksekusi (s)"],
        row["Silhouette Score"],
        marker="o",
        color=scatter_colors[idx],
        s=200,
        edgecolors="black",
        linewidths=0.8,
        zorder=5,
        label=sc_label,
    )
    ax2.annotate(
        sc_label,
        xy=(row["Waktu Eksekusi (s)"], row["Silhouette Score"]),
        xytext=(10, 8),
        textcoords="offset points",
        fontsize=10,
        fontweight="bold",
        color="#222222",
    )

ax2.set_title(
    "Tradeoff: Waktu Eksekusi vs Silhouette Score",
    fontsize=14,
    fontweight="bold",
    pad=14,
)
ax2.set_xlabel("Rata-rata Waktu Eksekusi (detik)", fontsize=12, labelpad=8)
ax2.set_ylabel("Rata-rata Silhouette Score", fontsize=12, labelpad=8)

ax2.tick_params(axis="x", labelsize=10)
ax2.tick_params(axis="y", labelsize=10)
ax2.grid(linestyle="--", alpha=0.4, color="gray")
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)

ax2.legend(
    title="Metode",
    title_fontsize=10,
    fontsize=10,
    loc="best",
    framealpha=0.85,
)

fig2.tight_layout()
fig2.savefig("Grafik_Tradeoff.png", dpi=300, bbox_inches="tight")
plt.close(fig2)

print("Kedua grafik berhasil disimpan:")
print("  -> Grafik_Akurasi.png")
print("  -> Grafik_Tradeoff.png")

Kedua grafik berhasil disimpan:
  -> Grafik_Akurasi.png
  -> Grafik_Tradeoff.png
